# Lab 5 — Delta Storage Optimization: OPTIMIZE, Z-ORDER & VACUUM

**Goal:** Practice file compaction, data layout optimization, and understand retention-safe cleanup.

**Safety:** `VACUUM` permanently removes unreferenced data files beyond the retention period and can limit time travel. This notebook uses a dedicated lab table. Do not reduce retention or disable safety checks for this exercise.

In [0]:

%sql
CREATE OR REPLACE TABLE workspace.lab_db.optimization_lab
USING DELTA
AS
SELECT
    id,
    concat('customer_', CAST(id AS STRING)) AS customer_name,
    date_add(DATE '2024-01-01', CAST(id % 365 AS INT)) AS signup_date,
    CAST((id % 1000) + 1 AS DOUBLE) AS amount
FROM range(0, 10000);


In [0]:
%sql
DESCRIBE DETAIL workspace.lab_db.optimization_lab;

In [0]:
%sql
-- Inspect the table before optimization
SELECT COUNT(*) AS rows, COUNT(DISTINCT customer_name) AS distinct_customers
FROM workspace.lab_db.optimization_lab;

In [0]:
%sql
OPTIMIZE workspace.lab_db.optimization_lab
ZORDER BY (customer_name, signup_date);

In [0]:
%sql
DESCRIBE DETAIL workspace.lab_db.optimization_lab;

## VACUUM

The guide uses a seven-day retention period. On managed Databricks environments, keep the default retention safety checks enabled and do not run `VACUUM ... RETAIN 0 HOURS`. If your workspace blocks VACUUM due to permissions or retention policy, record that result and continue.

In [0]:
%sql
VACUUM workspace.lab_db.optimization_lab RETAIN 168 HOURS;

In [0]:
%sql
SELECT COUNT(*) AS verified_rows
FROM workspace.lab_db.optimization_lab;